# Wan2.1 Cinematic Image-to-Video Studio
### One-run Google Colab workflow

Upload one of your scene images, describe the motion and camera movement, and generate an MP4 with Wan2.1 VACE 1.3B.

**Before Run all:** In Colab choose `Runtime → Change runtime type → T4 GPU` (or another available GPU). Free GPU availability is controlled by Google and is not guaranteed. The model is large; generation may take a while and can run out of memory on some sessions.

This notebook uses the official Wan2.1 VACE 1.3B reference-to-video workflow, which accepts still reference images. Official project: https://github.com/Wan-Video/Wan2.1


In [ ]:
import os, subprocess, sys, torch
print('Python:', sys.version)
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. In Colab select Runtime → Change runtime type → GPU, then run all cells again.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))


In [ ]:
from pathlib import Path
WORK = Path('/content')
REPO = WORK / 'Wan2.1'
if REPO.exists():
    subprocess.run(['bash','-lc', f'cd {REPO} && git pull'], check=True)
else:
    subprocess.run(['git','clone','https://github.com/AffanNamu/Wan2.1.git', str(REPO)], check=True)
os.chdir(REPO)
print('Project ready:', REPO)


In [ ]:
# Install project dependencies while avoiding flash-attn compilation (not suitable for standard free Colab setup).
req = (REPO / 'requirements.txt').read_text().splitlines()
filtered = [line for line in req if 'flash_attn' not in line.lower() and line.strip() and not line.lstrip().startswith('#')]
Path('/content/wan_colab_requirements.txt').write_text('\n'.join(filtered) + '\n')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', '/content/wan_colab_requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
print('Dependencies installed.')


In [ ]:
from huggingface_hub import snapshot_download
MODEL_DIR = '/content/Wan2.1-VACE-1.3B'
print('Downloading Wan-AI/Wan2.1-VACE-1.3B. This is a large download; wait for it to finish.')
snapshot_download(repo_id='Wan-AI/Wan2.1-VACE-1.3B', local_dir=MODEL_DIR)
print('Model ready:', MODEL_DIR)


## 1. Upload your scene image
Choose a clear image from your film scenes. Use one image per generation.

In [ ]:
from google.colab import files
from PIL import Image
import shutil
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No image uploaded.')
image_name = next(iter(uploaded))
image_path = str(Path('/content') / image_name)
img = Image.open(image_path).convert('RGB')
print('Loaded:', image_name, img.size)
display(img)


## 2. Describe the shot
Write what should move, what the camera should do, the atmosphere, and what must remain consistent. Example: `The man slowly turns toward the doorway as dust drifts through a shaft of warm light. The camera makes a very slow cinematic push-in. Preserve the character's face, clothing, and the original composition.`

In [ ]:
prompt = input('Describe the motion and camera movement for this shot: ').strip()
if not prompt:
    raise ValueError('Please enter a motion prompt and rerun this cell.')
negative_prompt = 'low quality, blurry, flicker, jitter, distorted face, deformed hands, extra limbs, changing clothes, identity change, warped background, text, watermark'
print('Prompt saved.')


## 3. Generate the video
Recommended starting point: 480p, 81 frames (about 5 seconds at the model frame rate). Generation may take several minutes or longer.

In [ ]:
import shlex, time
output_path = '/content/wan_scene_output.mp4'
cmd = [sys.executable, 'generate.py', '--task', 'vace-1.3B', '--size', '832*480', '--ckpt_dir', MODEL_DIR, '--src_ref_images', image_path, '--prompt', prompt, '--frame_num', '81', '--offload_model', 'True', '--t5_cpu', '--save_file', output_path]
print('Starting Wan2.1...')
print(' '.join(shlex.quote(x) for x in cmd))
started = time.time()
result = subprocess.run(cmd, cwd=str(REPO))
if result.returncode != 0:
    raise RuntimeError('Wan2.1 generation failed. Check the error output above. If it is CUDA out-of-memory, restart with a higher-memory GPU if available.')
if not Path(output_path).exists():
    raise FileNotFoundError('Generation ended but the MP4 was not found.')
print('Finished in', round((time.time()-started)/60, 1), 'minutes.')


## 4. Preview and download
Your MP4 will also remain in the Colab session's `/content` folder until the runtime is reset.

In [ ]:
from IPython.display import Video, display
display(Video(output_path, embed=True))
files.download(output_path)
